# peS2o download (allenai/peS2o)

- Stream and save a local 500k subset for scaling tests.
- Optional filters: `SOURCE_SUBSTR` and `MIN_WORDS`.
- Output as Parquet shards.

In [ ]:
!pip -q install datasets pyarrow tqdm

In [ ]:
import gc
import time
import hashlib
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset
from tqdm.auto import tqdm

# =========================
# CONFIG
# =========================
DATASET_NAME = "allenai/peS2o"
DATASET_CONFIG = "v2"
SPLIT = "train"

TARGET_DOCS = 500_000

# Optional filters
SOURCE_SUBSTR = ""  # e.g., "s2orc" (empty = no filter)
MIN_WORDS = 0

READ_BATCH_SIZE = 256
ROWS_PER_FILE = 5_000

OUT_DIR = Path("..").resolve() / "data" / "pes2o_500k"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA = pa.schema([
    ("doc_id", pa.int64()),
    ("source", pa.string()),
    ("text", pa.string()),
    ("n_words", pa.int32()),
    ("n_chars", pa.int32()),
    ("exact_hash", pa.string()),
])

def exact_md5(text: str) -> str:
    return hashlib.md5(text.encode("utf-8", errors="ignore")).hexdigest()

def batched_iter(iterable, batch_size):
    batch = []
    for x in iterable:
        batch.append(x)
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

def write_chunk(rows, part_idx: int, out_dir: Path):
    table = pa.Table.from_pylist(rows, schema=SCHEMA)
    out_path = out_dir / f"part-{part_idx:05d}.parquet"
    pq.write_table(table, out_path, compression="zstd")
    return out_path

dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=SPLIT,
    streaming=True,
)

buffer = []
part_idx = 0
kept = 0
seen = 0
t0 = time.perf_counter()

for ex_batch in batched_iter(dataset, READ_BATCH_SIZE):
    seen += len(ex_batch)
    for ex in ex_batch:
        src = str(ex.get("source", "")).lower()
        if SOURCE_SUBSTR and SOURCE_SUBSTR not in src:
            continue
        text = ex.get("text")
        if not text or not isinstance(text, str):
            continue
        n_words = len(text.split())
        if n_words < MIN_WORDS:
            continue
        buffer.append({
            "doc_id": kept,
            "source": src,
            "text": text,
            "n_words": n_words,
            "n_chars": len(text),
            "exact_hash": exact_md5(text),
        })
        kept += 1
        if kept >= TARGET_DOCS:
            break

    while len(buffer) >= ROWS_PER_FILE:
        write_chunk(buffer[:ROWS_PER_FILE], part_idx, OUT_DIR)
        buffer = buffer[ROWS_PER_FILE:]
        part_idx += 1
        gc.collect()

    if kept and kept % 10_000 < READ_BATCH_SIZE:
        elapsed = time.perf_counter() - t0
        print(f"[kept={kept:,}] [seen={seen:,}] [elapsed={elapsed/60:.1f} min]")

    if kept >= TARGET_DOCS:
        break

if buffer:
    write_chunk(buffer, part_idx, OUT_DIR)

elapsed = time.perf_counter() - t0
print(f"Done. kept={kept:,}, seen={seen:,}, elapsed={elapsed/60:.2f} min")
print("Output dir:", OUT_DIR)

In [ ]:
def dir_size_bytes(path: Path) -> int:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

def count_rows_in_parquet_dir(parquet_dir: Path) -> int:
    total = 0
    for fp in sorted(parquet_dir.glob("part-*.parquet")):
        pf = pq.ParquetFile(fp)
        total += pf.metadata.num_rows
    return total

print("Rows:", count_rows_in_parquet_dir(OUT_DIR))
print("Size GB:", round(dir_size_bytes(OUT_DIR) / (1024**3), 4))